# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03.07a · Reporte local de comparación de modelos

Consulta automáticamente la publicación 03_07 en Google Drive, la compara con el estado local por fecha y SHA-256, descarga solo resultados cuando corresponde y publica tablas, figuras y un informe Markdown crítico. Usa OAuth de solo lectura, no Drive Desktop; no extrae pesos y no abre test.

Balanced accuracy evita que el desbalance de clases oculte el desempeño de ANY_DAMAGE [1], mientras AUPRC aporta una lectura apropiada para categorías minoritarias [2]. Las pruebas y decisiones permanecen separadas de test para evitar sesgo de selección [3]. La síntesis presenta todas las métricas y explicita incertidumbre, capacidad de revisión y limitaciones; no convierte un orden lexicográfico en superioridad universal.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from moderacion_peru.notebook_ui import notebook_progress, run_with_progress, show_callout, show_command, show_result, show_summary, show_table
OPERATIONAL_PROMPT=ROOT/'config/prompt_operacional_ollama_v3_2.md'
if not OPERATIONAL_PROMPT.is_file():
    raise FileNotFoundError(f'Falta el prompt operacional vigente: {OPERATIONAL_PROMPT}')
show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local', 'prompt_operacional': OPERATIONAL_PROMPT}, tone='success')


## Sincronización automática y verificable desde Google Drive

La primera ejecución requiere un cliente OAuth de tipo **aplicación de escritorio** para la Google Drive API. Descargue su JSON como `config/google_drive_oauth_client.json`; esta ruta y el token `.secrets/google_drive_token.json` están ignorados por Git. Al ejecutar, el navegador solicitará una sola vez acceso de solo lectura. Las ejecuciones posteriores comparan el manifiesto remoto con el local y solo descargan una publicación nueva o diferente. Si aún no se configuró OAuth, la celda conserva el reporte local y muestra el paso pendiente. La preparación está documentada en `docs/GOOGLE_DRIVE_03_07A.md`.

In [ ]:
from moderacion_peru.comparison_reporting import synchronize_google_drive_results, synchronize_latest_local_results

# OAuth usa permiso Drive de solo lectura. La primera autorización abre el navegador;
# después reutiliza el token local ignorado por Git. Nunca usa Drive Desktop.
AUTO_SYNC_FROM_GOOGLE_DRIVE=True
INTERACTIVE_GOOGLE_DRIVE_AUTH=True
SEARCH_ROOTS=(
    ROOT/'resultados/sincronizados/03_07',
    ROOT/'resultados/modelos',
)
drive_sync_result=None
if AUTO_SYNC_FROM_GOOGLE_DRIVE:
    try:
        drive_sync_result=run_with_progress(
            'Consulta y sincronización automática desde Google Drive',
            synchronize_google_drive_results,
            ROOT,
            interactive_auth=INTERACTIVE_GOOGLE_DRIVE_AUTH,
            progress_unit='publicación',
        )
        sync_result=drive_sync_result
        show_result('Estado remoto de Google Drive',drive_sync_result,tone='success')
    except (FileNotFoundError,PermissionError) as exc:
        show_callout(
            'Autorización de Drive pendiente',
            f'{exc} Se continuará con los resultados locales; complete la autorización y repita esta celda para activar la actualización remota.',
            tone='warning',
        )
        sync_result=run_with_progress(
            'Selección de la comparación local más reciente',
            synchronize_latest_local_results,
            ROOT,
            search_roots=SEARCH_ROOTS,
            progress_unit='fuente',
        )
else:
    sync_result=run_with_progress(
        'Selección de la comparación local más reciente',
        synchronize_latest_local_results,
        ROOT,
        search_roots=SEARCH_ROOTS,
        progress_unit='fuente',
    )
show_result('Resultados 03_07 sincronizados localmente',sync_result,tone='success')

## Reporte tabular, gráfico y crítico

In [ ]:
from IPython.display import Image, display
from moderacion_peru.comparison_reporting import generate_comparison_report

report_result=run_with_progress(
    'Tablas, figuras y análisis crítico',
    generate_comparison_report,
    sync_result['comparison_path'],
    freeze_path=sync_result.get('freeze_path'),
    test_path=sync_result.get('test_path'),
    output_dir=ROOT/'resultados/modelos',
    generate_figures=True,
    progress_unit='reporte',
)
show_summary('Reporte reproducible 03_07a',{
    'seleccionado':report_result['selected_id'],
    'estado_inferencial':report_result['winner_status'],
    'test_disponible':report_result['test_available'],
    'reporte_markdown':report_result['report_path'],
    'tablas':report_result['table_paths'],
    'figuras':report_result['figure_paths'],
},tone='success')
show_table('Ranking global completo',report_result['global_rows'],max_rows=40)
show_table('Seleccionado por categoría',report_result['selected_category_rows'],max_rows=10)
show_table('Mejores candidatos por categoría',report_result['category_winners'],max_rows=10)
for figure_path in report_result['figure_paths']:
    display(Image(filename=figure_path,width=1050))

## Referencias

[1] K. H. Brodersen, C. S. Ong, K. E. Stephan, et al., "The Balanced Accuracy and Its Posterior Distribution," in Proc. 20th Int. Conf. Pattern Recognition, 2010, pp. 3121–3124, doi: 10.1109/ICPR.2010.764.

[2] T. Saito and M. Rehmsmeier, "The Precision-Recall Plot Is More Informative than the ROC Plot When Evaluating Binary Classifiers on Imbalanced Datasets," PLOS ONE, vol. 10, no. 3, Art. no. e0118432, 2015, doi: 10.1371/journal.pone.0118432.

[3] G. C. Cawley and N. L. C. Talbot, "On Over-Fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation," J. Mach. Learn. Res., vol. 11, pp. 2079–2107, 2010.